## Tiktok Ads

#### Useful links:
- [Commercial Content API](https://developers.tiktok.com/products/commercial-content-api)
- [Commercial Content API - Getting Started](https://developers.tiktok.com/doc/commercial-content-api-getting-started)
- [Commercial Content API - Documentation](https://developers.tiktok.com/doc/commercial-content-api-query-ads?enter_method=left_navigation)
- [API Supported Countries](https://developers.tiktok.com/doc/commercial-content-api-supported-countries?enter_method=left_navigation)
- [Tiktok Ads Library](https://library.tiktok.com/ads)


In [ ]:
# requirements
%pip install pandas requests python-dotenv jmespath

In [ ]:
import os
import json
import glob
import pandas as pd
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [52]:
# environment variables
load_dotenv()
TIKTOK_CLIENT_KEY = os.getenv("TIKTOK_CLIENT_KEY")
TIKTOK_CLIENT_SECRET = os.getenv("TIKTOK_CLIENT_SECRET")
FILEPATH = os.getenv("FILEPATH")


#### Available Endpoints

- Query Ads: https://open.tiktokapis.com/v2/research/adlib/ad/query/
- Query Advertisers:     
https://open.tiktokapis.com/v2/research/adlib/advertiser/query/
- Query Ad Details: https://open.tiktokapis.com/v2/research/adlib/ad/detail/
- Query Ad Report: https://open.tiktokapis.com/v2/research/adlib/ad/report/
- Query Commercial Content: https://open.tiktokapis.com/v2/research/adlib/commercial_content/query/

#### API Connection

In [109]:
# GET BEARER TOKEN
token_resp = requests.post(
    url='https://open.tiktokapis.com/v2/oauth/token/',
    data={
        'client_key': TIKTOK_CLIENT_KEY,
        'client_secret': TIKTOK_CLIENT_SECRET,
        'grant_type': 'client_credentials'
    }
)
token_resp.status_code
token_resp = token_resp.json()
token = token_resp['access_token']

In [4]:
# BASE URL
base_url = "https://open.tiktokapis.com/v2/research/adlib"

In [ ]:
payload_example = {
    "filters": {
        "advertiser_business_ids": [3847236290405, 319282903829],
        "ad_published_date_range": {
            "min": "20210102", #YYYYMMDD
            "max": "20210109" #YYYYMMDD
        },
        "country_code": "FR",
        "unique_users_seen_size_range": {
            "min": "10K",
            "max": "1M"
        }
    },
    "search_term": "mobile games",
    "max_count": 20
}

In [5]:
# API requests
def request_to_api(access_token, endpoint_url, payload):
    headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
    response = requests.post(endpoint_url, headers=headers, json=payload)
    return response.json() # ads can be found in data.ads of the response


### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


In [6]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [7]:
display(pd.json_normalize(response["data"]["ads"]))

,ad.first_shown_date,ad.id,ad.last_shown_date,ad.reach.unique_users_seen,ad.status,ad.status_statement,ad.videos,advertiser.business_id,advertiser.business_name,advertiser.paid_for_by,ad.image_urls
0,20260205,1856300880717906,20260209,400K-500K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...,Dentsu Portugal,NaN
1,20260204,1856202347004977,20260207,10K-100K,active,N/A,[],7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,[https://p16-csp-adlib-site-sign-sg.ibyteimg.c...
2,20260204,1856202199672849,20260207,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
3,20260204,1856201982884961,20260207,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
4,20260206,1856369572974897,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
5,20260204,1856201371754513,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
6,20260203,1856111493198082,20260206,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,7602632199019610129,7602632240870195232,7602632240870195232,NaN
7,20260201,1855928238414913,20260203,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,7466211512521326608,7466211550051696662,7466211550051696662,NaN
8,20260203,1856008614710417,20260204,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,7049703761119281921,MED&CR - SERVIÇOS DE GESTÃO DE CARTÕES DE SAÚD...,GeroDigital,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
9,20260206,1856366133135521,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7321303080748957698,TRIPSOFT LIMITED,TRIPSOFT LIMITED,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...


**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


In [8]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20230101", 
            "max": "20251231"
        },
        "country_code": "PT"
    },
    "search_term": "política",
}
response = request_to_api(token, full_url, payload)

In [9]:
df = pd.json_normalize(response['data']['ads'])
display(df[["ad.id", "ad.first_shown_date", "ad.last_shown_date", "ad.status"]])

,ad.id,ad.first_shown_date,ad.last_shown_date,ad.status
0,1817530779907122,20241205,20241211,active
1,1817530779839537,20241205,20241211,active
2,1817530779834417,20241205,20241211,active
3,1815696351242258,20241114,20241120,active
4,1832932515638481,20250523,20250524,active
5,1826212633685266,20250310,20250311,active
6,1838280085636113,20250721,20250722,active
7,1798767621083153,20240511,20240512,active
8,1848754976858242,20251114,20251115,active
9,1850726375153665,20251206,20251207,active


### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


In [10]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "GB"
    },
    "search_term": "health"
}
response = request_to_api(token, full_url, payload)

In [11]:
response["data"]["ads"][0]

{'ad': {'videos': [{'cover_image_url': 'https://p16-sign-va.tiktokcdn.com/tos-maliva-p-0068c799-us/oQqkkkgrBId7LE8mAKiCpVIKoAfO09BIiJwMCo~tplv-noop.image?dr=18692&refresh_token=e6e1fd56&x-expires=1770775139&x-signature=gobAvRrfQm7iCJ7JDJi3ZwXq8d8%3D&t=9276707c&ps=14f1eb3e&shp=9e36835a&shcp=ec394d43&idc=sg1&VideoID=v12044gd0000d52965fog65slj4gl85g',
    'url': 'https://v16m.tiktokcdn.com/710221f1ad66336140986a1457e63625/698be263/video/tos/maliva/tos-maliva-ve-0068c799-us/oopopIKEBkxmkAkowif7AqCTKirMT0BCM8LItE/?a=475769&bti=ODM6N2YuLjY6&ch=0&cr=0&dr=1&cd=0%7C0%7C0%7C0&cv=1&br=366&bt=183&cs=0&ds=6&ft=.NpOcInz7ThkuxPPXq8Zmo&mime_type=video_mp4&qs=5&rc=Zjw6ODpmZmZlO2c2aDVmNUBpajtqbXY5cjg8ODMzZzczNEA0Ml8xMF40Xy0xMjQ1MTVfYSMyaTFpMmRjMy9hLS1kMS9zcw%3D%3D&vvpl=1&l=20260210195853C8D9DF7900DB722111F6&btag=e000b0000&cc=3'}],
  'first_shown_date': '20260201',
  'id': 1855926421908529,
  'last_shown_date': '20260209',
  'reach': {'unique_users_seen': '700K-800K'},
  'status': 'active',
  'status_sta

In [12]:
endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1855926421908529
}
response = request_to_api(token, full_url, payload)

In [14]:
print(json.dumps(response["data"]["ad"], indent=4))

{
    "last_shown_date": "20260209",
    "reach": {
        "unique_users_seen_by_country": {
            "CH": "16K",
            "DE": "59K",
            "NO": "8K",
            "PT": "23K",
            "AT": "14K",
            "ES": "24K",
            "PL": "35K",
            "CZ": "30K",
            "GB": "335K",
            "HU": "19K",
            "SE": "33K",
            "BE": "8K",
            "FI": "6K",
            "FR": "44K",
            "IE": "84K",
            "IT": "18K",
            "NL": "10K",
            "DK": "27K"
        },
        "unique_users_seen": "700K-800K"
    },
    "status": "active",
    "status_statement": "N/A",
    "videos": [
        {
            "url": "https://v77.tiktokcdn.com/3dadebc9a3d7ec5c837a46a8838ac5cc/698be29e/video/tos/maliva/tos-maliva-ve-0068c799-us/oopopIKEBkxmkAkowif7AqCTKirMT0BCM8LItE/?a=475769&bti=ODM6N2YuLjY6&ch=0&cr=0&dr=1&cd=0%7C0%7C0%7C0&cv=1&br=366&bt=183&cs=0&ds=6&ft=.NpOcInz7ThwUxPPXq8Zmo&mime_type=video_mp4&qs=5&rc=Zjw6ODp

**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



In [15]:
endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1781554420341777
}
response = request_to_api(token, full_url, payload)

In [16]:
print(json.dumps(response["data"]["ad"], indent=4))

{
    "image_urls": [],
    "last_shown_date": "20260209",
    "reach": {
        "unique_users_seen": "10K-100K",
        "unique_users_seen_by_country": {
            "DE": "0-1K",
            "DK": "0-1K",
            "ES": "2K",
            "FR": "0-1K",
            "GB": "0-1K",
            "PT": "5K",
            "BE": "2K",
            "CZ": "2K",
            "IE": "0-1K",
            "IT": "2K",
            "NL": "0-1K",
            "AT": "0-1K",
            "CH": "0-1K",
            "HU": "5K",
            "NO": "0-1K",
            "SE": "0-1K",
            "FI": "0-1K",
            "PL": "4K",
            "RO": "11K"
        }
    },
    "status": "active",
    "status_statement": "N/A",
    "videos": [
        {
            "cover_image_url": "https://p16-sign-va.tiktokcdn.com/tos-maliva-p-0068/4e35f3004f6c431fb9e29ba3692a6e32_1698969797~tplv-noop.image?dr=18692&refresh_token=cfca7efa&x-expires=1770775296&x-signature=vpw5yINsbm5u5yFotyLyl%2BKbGXU%3D&t=9276707c&ps=14f1eb3e&sh

**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


In [17]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "advertiser_business_ids": [7112381089636860674],
        "ad_published_date_range": {
            "min": "20250101", 
            "max": "20260208"
        },
        "country_code": "PT"
    }
}
response = request_to_api(token, full_url, payload)

In [19]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id", "advertiser.business_id", "advertiser.business_name"]])

,ad.id,advertiser.business_id,advertiser.business_name
0,1838428214566050,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
1,1851754570607698,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
2,1832027317494834,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
3,1832027414490177,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
4,1832087890373857,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
5,1850046514885794,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
6,1850047036683537,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
7,1827114203869345,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
8,1850050764355810,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
9,1827134776032321,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."


**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


In [20]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260208"
        },
        "country_code": "GB"
    },
    "search_term": "health"
}
response = request_to_api(token, full_url, payload)

In [21]:
display(pd.json_normalize(response["data"]["ads"]))

,ad.status_statement,ad.videos,ad.first_shown_date,ad.id,ad.last_shown_date,ad.reach.unique_users_seen,ad.status,advertiser.business_id,advertiser.business_name,advertiser.paid_for_by,ad.image_urls
0,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,1855926421908529,20260209,700K-800K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN
1,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,1855925693834498,20260209,700K-800K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN
2,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260203,1856082551254081,20260206,600K-700K,active,7595466585436110865,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
3,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260204,1856198460606609,20260206,500K-600K,active,7595466585436110865,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
4,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260204,1856204131723394,20260208,500K-600K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN
5,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,1855925693834514,20260209,400K-500K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN
6,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260208,1856520026135778,20260209,400K-500K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN
7,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260201,1855900590386322,20260206,300K-400K,active,7595466585436110865,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
8,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260203,1856082824746242,20260206,300K-400K,active,7595466585436110865,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
9,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,1855925693834530,20260209,200K-300K,active,7115383334368494338,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,NaN


**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [22]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "GB"
    },
    "search_term": "health"
}
response = request_to_api(token, full_url, payload)

In [23]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id", "ad.first_shown_date", "ad.last_shown_date", "ad.reach.unique_users_seen"]])

,ad.id,ad.first_shown_date,ad.last_shown_date,ad.reach.unique_users_seen
0,1855926421908529,20260201,20260209,700K-800K
1,1855925693834498,20260201,20260209,700K-800K
2,1856082551254081,20260203,20260206,600K-700K
3,1856198460606609,20260204,20260206,500K-600K
4,1856204131723394,20260204,20260208,500K-600K
5,1855925693834514,20260201,20260209,400K-500K
6,1855900590386322,20260201,20260206,300K-400K
7,1856082824746242,20260203,20260206,300K-400K
8,1855925693834530,20260201,20260209,200K-300K
9,1856067815108993,20260203,20260209,200K-300K


### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



In [24]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [25]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id","advertiser.business_id","advertiser.business_name"]])

,ad.id,advertiser.business_id,advertiser.business_name
0,1856300880717906,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...
1,1856202347004977,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
2,1856202199672849,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
3,1856201982884961,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
4,1856369572974897,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
5,1856201371754513,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A."
6,1856111493198082,7602632199019610129,7602632240870195232
7,1855928238414913,7466211512521326608,7466211550051696662
8,1856008614710417,7049703761119281921,MED&CR - SERVIÇOS DE GESTÃO DE CARTÕES DE SAÚD...
9,1856366133135521,7321303080748957698,TRIPSOFT LIMITED


**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


In [26]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [27]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id", "advertiser.paid_for_by"]])

,ad.id,advertiser.paid_for_by
0,1856300880717906,Dentsu Portugal
1,1856202347004977,BYD - Boost Your Digital
2,1856202199672849,BYD - Boost Your Digital
3,1856201982884961,BYD - Boost Your Digital
4,1856369572974897,BYD - Boost Your Digital
5,1856201371754513,BYD - Boost Your Digital
6,1856111493198082,7602632240870195232
7,1855928238414913,7466211550051696662
8,1856008614710417,GeroDigital
9,1856366133135521,TRIPSOFT LIMITED


**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

In [28]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [29]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id",  "ad.first_shown_date","ad.last_shown_date"]])

,ad.id,ad.first_shown_date,ad.last_shown_date
0,1856300880717906,20260205,20260209
1,1856202347004977,20260204,20260207
2,1856202199672849,20260204,20260207
3,1856201982884961,20260204,20260207
4,1856369572974897,20260206,20260209
5,1856201371754513,20260204,20260209
6,1856111493198082,20260203,20260206
7,1855928238414913,20260201,20260203
8,1856008614710417,20260203,20260204
9,1856366133135521,20260206,20260209


**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



In [30]:
endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1684129375777841
}
response = request_to_api(token, full_url, payload)

In [32]:
print(json.dumps(response["data"], indent=4))

{
    "ad": {
        "first_shown_date": "20221001",
        "id": 1684129375777841,
        "last_shown_date": "20260209",
        "reach": {
            "unique_users_seen": "-",
            "unique_users_seen_by_country": {}
        },
        "status": "inactive",
        "status_statement": "Removed from TikTok due to a violation of TikTok's terms"
    },
    "ad_group": {
        "targeting_info": {
            "video_interactions": "",
            "age": {
                "25-34": true,
                "35-44": true,
                "45-54": true,
                "55+": true,
                "13-17": true,
                "18-24": true
            },
            "audience_targeting": "No",
            "country": [],
            "creator_interactions": "",
            "gender": {
                "other_genders": false,
                "female": true,
                "male": false
            },
            "interest": "News & Entertainment, Baby & Kids Products, Beauty & Persona

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [33]:
# Use this cell to develop an example where a request for ads from a campaign is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1781554420341777
}
response = request_to_api(token, full_url, payload)

In [34]:
print(json.dumps(response["data"], indent=4))

{
    "ad": {
        "status_statement": "N/A",
        "videos": [
            {
                "cover_image_url": "https://p16-sign-va.tiktokcdn.com/tos-maliva-p-0068/4e35f3004f6c431fb9e29ba3692a6e32_1698969797~tplv-noop.image?dr=18692&refresh_token=9c72b96e&x-expires=1770775585&x-signature=jPGV3fUVeDdyUMYCODNx1r3Qk0c%3D&t=9276707c&ps=14f1eb3e&shp=9e36835a&shcp=ec394d43&idc=sg1&VideoID=v09025g40000cl23g1nog65ncan5gc00",
                "url": "https://v58.tiktokcdn.com/video/tos/useast2a/tos-useast2a-ve-0068c003/oEQ54AUEMYATBEVYMmvAZODHliUtqiQIiSQB3/?a=475769&bti=ODM6N2YuLjY6&ch=0&cr=0&dr=1&cd=0%7C0%7C0%7C0&cv=1&br=1960&bt=980&cs=0&ds=6&ft=.NpOcInz7ThtsxPPXq8Zmo&mime_type=video_mp4&qs=0&rc=NGg4PDlmZDlpZzs1ZjM0ZEBpM2Y4ZHE5cjQ2bzMzNzgzM0AxMC01NGMxNjAxNS9eX2EvYSMtZGtgMmRrZC9gLS1kLzZzcw%3D%3D&vvpl=1&l=202602102005576BFA23BA4016B5250CD1&VExpiration=1770775585&VSignature=BSmz6wi4co4htFDi64gUUw&btag=e000b8000&cc=14"
            }
        ],
        "first_shown_date": "20231104",
        

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [ ]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260205"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
total_runs = 5 
for idx in range(total_runs):
    index = idx + 1
    filepath = f"{FILEPATH}/eu-tiktok-ads-question-24-run-{index}-{datetime.now().isoformat()}"

    # DEVELOP YOUR CODE HERE
    response = request_to_api(token, full_url, payload) # ads can be found in data.ads of the response

    with open(filepath, "w") as fout:
        json.dump(response["data"]["ads"], fout)
    sleep(180)

In [64]:
files = sorted(glob.glob(f"{FILEPATH}/eu-tiktok-ads-question-24*"))
dfs = [pd.read_json(f) for f in files]
dfs = [pd.json_normalize(df.to_dict(orient="records")) for df in dfs]
id_sets = [set(df['ad.id']) for df in dfs]

all_same_ids = all(s == id_sets[0] for s in id_sets)

print("All requests returned the same response:", all_same_ids)

All requests returned the same response: True


**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [ ]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"


In [75]:
# date range
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260210"
        }
    },
    "search_term": "health"
}
filepath = f"{FILEPATH}/eu-tiktok-ads-question-25-run-1-ad_published_date_range-{datetime.now().isoformat()}"
response = request_to_api(token, full_url, payload)
data = response["data"]["ads"]
with open(filepath, "w") as fout:
    json.dump(data, fout)

In [79]:
df = pd.json_normalize(data)
df["ad.first_shown_date"] = pd.to_datetime(df["ad.first_shown_date"])
df["ad.last_shown_date"] = pd.to_datetime(df["ad.last_shown_date"])
start = pd.Timestamp("2026-02-01")
end = pd.Timestamp("2026-02-10")
result = (
    df["ad.first_shown_date"].between(start, end) &
    df["ad.last_shown_date"].between(start, end)
).all()

print("All ads ran between the set date:", result)

All ads ran between the set date: True


In [93]:
# country
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260205"
        },
        "country_code": "GB",
    },
    "search_term": "health"
}
filepath = f"{FILEPATH}/eu-tiktok-ads-question-25-run-2-country_code-{datetime.now().isoformat()}"
response = request_to_api(token, full_url, payload)
data = response["data"]["ads"]
with open(filepath, "w") as fout:
    json.dump(data, fout)

In [96]:
display(pd.json_normalize(data))

,ad.id,ad.last_shown_date,ad.reach.unique_users_seen,ad.status,ad.status_statement,ad.videos,ad.first_shown_date,advertiser.business_name,advertiser.paid_for_by,advertiser.business_id,ad.image_urls
0,1855925693834498,20260210,800K-900K,active,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,7115383334368494338,NaN
1,1855926421908529,20260210,700K-800K,active,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,7115383334368494338,NaN
2,1856082551254081,20260206,600K-700K,active,N/A,[{'url': 'https://v77.tiktokcdn.com/549c81d0b9...,20260203,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,7595466585436110865,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
3,1855925693834514,20260210,500K-600K,active,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,7115383334368494338,NaN
4,1856198460606609,20260206,500K-600K,active,N/A,[{'url': 'https://v77.tiktokcdn.com/7fb65df907...,20260204,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,7595466585436110865,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
5,1856204131723394,20260208,500K-600K,active,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260204,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,7115383334368494338,NaN
6,1855900590386322,20260206,300K-400K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260201,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,7595466585436110865,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
7,1856082824746242,20260206,300K-400K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,20260203,TPUC GLOBAL PTY LTD,TPUC GLOBAL PTY LTD,7595466585436110865,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
8,1856280549625474,20260210,300K-400K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,20260206,RECKITT BENCKISER HEALTH LIMITED,RECKITT BENCKISER HEALTH LIMITED,6876496852737327874,[https://p21-ad-sg.ibyteimg.com/origin/tos-ali...
9,1855925693834530,20260210,200K-300K,active,N/A,[{'cover_image_url': 'https://p16-sign-va.tikt...,20260201,FLO HEALTH UK LIMITED,FLO HEALTH UK LIMITED,7115383334368494338,NaN


In [100]:
# advertiser
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260208"
        },
        "advertiser_business_ids": [7115383334368494338]
    }
    
}
filepath = f"{FILEPATH}/eu-tiktok-ads-question-25-run-3-advertiser_business_ids-{datetime.now().isoformat()}"
response = request_to_api(token, full_url, payload)
data = response["data"]["ads"]
with open(filepath, "w") as fout:
    json.dump(data, fout)

In [102]:
df = pd.json_normalize(data)
result = df["advertiser.business_id"].eq(7115383334368494338).all()
print("All the ads are from the set advertiser:", result)

All the ads are from the set advertiser: True


In [104]:
# impressions
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260205"
        },
        "unique_users_seen_size_range": {"min": "100K", "max": "200K"},
    },
    "search_term": "health"
}
filepath = f"{FILEPATH}/eu-tiktok-ads-question-25-run-4-unique_users_seen_size_range-{datetime.now().isoformat()}"
response = request_to_api(token, full_url, payload)
data = response["data"]["ads"]
with open(filepath, "w") as fout:
    json.dump(data, fout)

In [105]:
df = pd.json_normalize(data)
df["ad.reach.unique_users_seen"]

0    100K-200K
1    100K-200K
2    100K-200K
3    100K-200K
4    100K-200K
5    100K-200K
6    100K-200K
7    100K-200K
8    100K-200K
9    100K-200K
Name: ad.reach.unique_users_seen, dtype: object

### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



In [35]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260209"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [36]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id",  "ad.first_shown_date","ad.last_shown_date"]])

,ad.id,ad.first_shown_date,ad.last_shown_date
0,1856300880717906,20260205,20260209
1,1856202347004977,20260204,20260207
2,1856202199672849,20260204,20260207
3,1856201982884961,20260204,20260207
4,1856369572974897,20260206,20260209
5,1856201371754513,20260204,20260209
6,1856651713597506,20260209,20260209
7,1856651207653394,20260209,20260209
8,1856651398853810,20260209,20260209
9,1856111493198082,20260203,20260206


**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*

In [37]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260209"
        },
        "country_code": "PT"
    },
    "search_term": "saúde"
}
response = request_to_api(token, full_url, payload)

In [38]:
display(pd.json_normalize(response["data"]["ads"]))

,ad.first_shown_date,ad.id,ad.last_shown_date,ad.reach.unique_users_seen,ad.status,ad.status_statement,ad.videos,advertiser.business_id,advertiser.business_name,advertiser.paid_for_by,ad.image_urls
0,20260205,1856300880717906,20260209,400K-500K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...,Dentsu Portugal,NaN
1,20260204,1856202347004977,20260207,10K-100K,active,N/A,[],7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,[https://p16-csp-adlib-site-sign-sg.ibyteimg.c...
2,20260204,1856202199672849,20260207,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
3,20260204,1856201982884961,20260207,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
4,20260206,1856369572974897,20260209,10K-100K,active,N/A,[{'url': 'https://v16m.tiktokcdn.com/6e294a95a...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
5,20260204,1856201371754513,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p16-sign-sg.tikt...,7112381089636860674,"PHARMACONTINENTE - SAÚDE E HIGIENE, S.A.",BYD - Boost Your Digital,NaN
6,20260209,1856651713597506,20260209,10K-100K,active,N/A,[{'url': 'https://v16m.tiktokcdn.com/a4321a67b...,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...,Dentsu Portugal,NaN
7,20260209,1856651207653394,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...,Dentsu Portugal,NaN
8,20260209,1856651398853810,20260209,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,6948730507886592770,PROCTER & GAMBLE PORTUGAL - PRODUTOS DE CONSUM...,Dentsu Portugal,NaN
9,20260203,1856111493198082,20260206,10K-100K,active,N/A,[{'cover_image_url': 'https://p19-common-sign-...,7602632199019610129,7602632240870195232,7602632240870195232,NaN


### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [44]:
endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1781554420341777
}
response = request_to_api(token, full_url, payload)

In [45]:
ad = {"ad_id": response["data"]["ad"]["id"], "reach_data":response["data"]["ad"]["reach"]}
print(json.dumps(ad, indent=4))

{
    "ad_id": 1781554420341777,
    "reach_data": {
        "unique_users_seen": "10K-100K",
        "unique_users_seen_by_country": {
            "PT": "5K",
            "BE": "2K",
            "CH": "0-1K",
            "CZ": "2K",
            "FR": "0-1K",
            "HU": "5K",
            "IT": "2K",
            "NL": "0-1K",
            "RO": "11K",
            "AT": "0-1K",
            "DE": "0-1K",
            "DK": "0-1K",
            "FI": "0-1K",
            "GB": "0-1K",
            "IE": "0-1K",
            "NO": "0-1K",
            "PL": "4K",
            "ES": "2K",
            "SE": "0-1K"
        }
    }
}


**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


In [46]:
endpoint_suffix = "ad/detail" 
fields = [
"ad.id",
"ad.first_shown_date",
"ad.last_shown_date",
"ad.status",
"ad.status_statement",
"ad.videos",
"ad.image_urls",
"ad.reach",
"advertiser.business_id",
"advertiser.business_name",
"advertiser.paid_for_by",
"advertiser.follower_count",
"advertiser.avatar_url",
"advertiser.profile_url",
"ad_group.targeting_info",
]
params = {
    "fields": ",".join(fields)
}
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "ad_id": 1796149731862530
}
response = request_to_api(token, full_url, payload)

In [47]:
ad = {"ad_id": response["data"]["ad"]["id"], "targeting_data":response["data"]["ad_group"]}
print(json.dumps(ad, indent=4))

{
    "ad_id": 1796149731862530,
    "targeting_data": {
        "targeting_info": {
            "audience_targeting": "No",
            "country": [
                "GB"
            ],
            "creator_interactions": "",
            "gender": {
                "female": true,
                "male": false,
                "other_genders": false
            },
            "interest": "News & Entertainment",
            "number_of_users_targeted": "5.5M-6.7M",
            "video_interactions": "",
            "age": {
                "18-24": true,
                "25-34": true,
                "35-44": true,
                "45-54": true,
                "55+": true,
                "13-17": false
            }
        }
    }
}


**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [120]:
endpoint_suffix = "ad/query" 
fields = [
    'ad.id',
    'ad.first_shown_date',
    'ad.last_shown_date',
    'ad.status',
    'ad.status_statement',
    'ad.videos',
    'ad.image_urls',
    'ad.reach',
    'advertiser.business_id',
    'advertiser.business_name',
    'advertiser.paid_for_by',
]
full_url = f"{base_url}/{endpoint_suffix}/?fields={','.join(fields)}"
payload = {
    "filters": {
        "ad_published_date_range": {
            "min": "20260201", 
            "max": "20260207"
        },
        "unique_users_seen_size_range": { "min": "10K", "max": "800K"}
    },
    "search_term": "health"
}
response = request_to_api(token, full_url, payload)

In [121]:
data = pd.json_normalize(response["data"]["ads"])
display(data[["ad.id",  "ad.reach.unique_users_seen"]])

,ad.id,ad.reach.unique_users_seen
0,1855926421908529,700K-800K
1,1856304976159841,600K-700K
2,1856082551254081,600K-700K
3,1855925693834514,500K-600K
4,1856386460215490,500K-600K
5,1856116042389666,500K-600K
6,1856198460606609,500K-600K
7,1856307193318513,500K-600K
8,1856204131723394,500K-600K
9,1856352403366978,400K-500K


**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*
